In [2]:
from pyspark.sql.functions import (
    col, trim, upper, when, current_timestamp,
    lit, to_date, round as spark_round
)
from pyspark.sql.types import DecimalType, IntegerType
from datetime import datetime

BRONZE_TABLE  = "Interac_Bronze.dbo.settlements"
SILVER_TABLE  = "silver_settlements"
SILVER_DB     = "Interac_Fabric_Workspace.Interac_Silver.dbo"
PIPELINE_NAME = "NB_05_Silver_Settlements"
BATCH_DATE    = datetime.now().strftime("%Y-%m-%d")

print(f"Silver Settlements Pipeline")
print(f"Started: {datetime.now()}")

StatementMeta(, da5f9714-a76f-4658-ba44-fc4a94656a7c, 4, Finished, Available, Finished, False)

Silver Settlements Pipeline
Started: 2026-05-06 00:23:15.443643


In [3]:
df_bronze = spark.read.table(BRONZE_TABLE)
total_bronze = df_bronze.count()
print(f"Bronze rows read: {total_bronze:,}")

dq_results = {}
dq_results["null_settlement_id"] = df_bronze.filter(col("settlement_id").isNull()).count()
dq_results["null_merchant_id"] = df_bronze.filter(col("merchant_id").isNull()).count()
dq_results["null_gross_amount"] = df_bronze.filter(col("gross_amount_cad").isNull()).count()
dq_results["negative_net_amount"] = df_bronze.filter(
    col("net_settlement_amount_cad").cast(DecimalType(18,2)) < 0).count()
dq_results["failed_settlements"] = df_bronze.filter(col("status") == "FAILED").count()
dq_results["disputed_settlements"] = df_bronze.filter(col("status") == "DISPUTED").count()
dq_results["reversed_settlements"] = df_bronze.filter(col("status") == "REVERSED").count()
dq_results["pending_settlements"] = df_bronze.filter(col("status") == "PENDING").count()

print("\nDQ CHECK RESULTS:")
print("-" * 45)
for check, count_val in dq_results.items():
    status = "⚠ FLAGGED" if count_val > 0 else "✓ PASSED"
    print(f"{check:<35} {count_val:>6,}  {status}")

StatementMeta(, da5f9714-a76f-4658-ba44-fc4a94656a7c, 5, Finished, Available, Finished, False)

Bronze rows read: 34,561

DQ CHECK RESULTS:
---------------------------------------------
null_settlement_id                       0  ✓ PASSED
null_merchant_id                         0  ✓ PASSED
null_gross_amount                        0  ✓ PASSED
negative_net_amount                      0  ✓ PASSED
failed_settlements                     721  ⚠ FLAGGED
disputed_settlements                   688  ⚠ FLAGGED
reversed_settlements                   691  ⚠ FLAGGED
pending_settlements                  2,059  ⚠ FLAGGED


In [4]:
df_quarantine = df_bronze.filter(
    col("settlement_id").isNull() |
    col("merchant_id").isNull() |
    col("gross_amount_cad").isNull()
)
quarantine_count = df_quarantine.count()

if quarantine_count > 0:
    (df_quarantine
        .withColumn("_quarantine_reason", lit("NULL_PRIMARY_KEY_OR_AMOUNT"))
        .withColumn("_quarantined_at", current_timestamp())
        .write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER_DB}.silver_settlements_quarantine"))
    print(f"Quarantined: {quarantine_count:,} records")

df_valid = df_bronze.filter(
    col("settlement_id").isNotNull() &
    col("merchant_id").isNotNull() &
    col("gross_amount_cad").isNotNull()
)

df_silver = (df_valid
    .withColumn("settlement_id",    trim(col("settlement_id")))
    .withColumn("merchant_id",      trim(col("merchant_id")))
    .withColumn("acquiring_bank",   trim(col("acquiring_bank")))
    .withColumn("currency",         upper(trim(col("currency"))))
    .withColumn("status",           upper(trim(col("status"))))
    .withColumn("settlement_date",
        to_date(col("settlement_date"), "yyyy-MM-dd"))
    .withColumn("value_date",
        to_date(col("value_date"), "yyyy-MM-dd"))
    .withColumn("transaction_count",
        col("transaction_count").cast(IntegerType()))
    .withColumn("gross_amount_cad",
        col("gross_amount_cad").cast(DecimalType(18, 2)))
    .withColumn("interchange_fees_cad",
        col("interchange_fees_cad").cast(DecimalType(18, 2)))
    .withColumn("network_fees_cad",
        col("network_fees_cad").cast(DecimalType(18, 2)))
    .withColumn("processing_fees_cad",
        col("processing_fees_cad").cast(DecimalType(18, 2)))
    .withColumn("chargeback_debit_cad",
        col("chargeback_debit_cad").cast(DecimalType(18, 2)))
    .withColumn("net_settlement_amount_cad",
        col("net_settlement_amount_cad").cast(DecimalType(18, 2)))
    .withColumn("total_fees_cad",
        spark_round(
            col("interchange_fees_cad") +
            col("network_fees_cad") +
            col("processing_fees_cad"), 2))
    .withColumn("fee_rate_pct",
        spark_round(
            col("interchange_fees_cad") /
            col("gross_amount_cad") * 100, 4))
    .withColumn("has_chargeback",
        when(col("chargeback_debit_cad") > 0, "Y").otherwise("N"))
    .withColumn("is_completed",
        when(col("status") == "COMPLETED", "Y").otherwise("N"))
    .withColumn("settlement_month",
        col("settlement_date").cast("string").substr(1, 7))
    .withColumn("_silver_loaded_at", current_timestamp())
    .withColumn("_pipeline_name",    lit(PIPELINE_NAME))
    .withColumn("_batch_date",       lit(BATCH_DATE))
    .drop("_ingested_at", "_source_file", "_lakehouse")
)

print(f"Valid records: {df_silver.count():,}")

StatementMeta(, da5f9714-a76f-4658-ba44-fc4a94656a7c, 6, Finished, Available, Finished, False)

Valid records: 34,561


In [5]:
(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.autoOptimize.optimizeWrite", "true")
    .saveAsTable(f"{SILVER_DB}.{SILVER_TABLE}"))

spark.sql(f"OPTIMIZE {SILVER_DB}.{SILVER_TABLE} ZORDER BY (settlement_date, merchant_id)")

final_count = spark.read.table(f"{SILVER_DB}.{SILVER_TABLE}").count()

print("\n" + "="*60)
print("SILVER SETTLEMENTS SUMMARY")
print("="*60)
print(f"Bronze rows in    : {total_bronze:,}")
print(f"Quarantined       : {quarantine_count:,}")
print(f"Silver rows out   : {final_count:,}")
print(f"Pass rate         : {round(final_count/total_bronze*100, 2)}%")
print(f"Table             : {SILVER_DB}.{SILVER_TABLE}")
print(f"Completed at      : {datetime.now()}")
print("="*60)

StatementMeta(, da5f9714-a76f-4658-ba44-fc4a94656a7c, 7, Finished, Available, Finished, False)


SILVER SETTLEMENTS SUMMARY
Bronze rows in    : 34,561
Quarantined       : 0
Silver rows out   : 34,561
Pass rate         : 100.0%
Table             : Interac_Fabric_Workspace.Interac_Silver.dbo.silver_settlements
Completed at      : 2026-05-06 00:24:21.805324
